In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.2 MB/s eta 0:00:00


In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange
from einops.layers.torch import Rearrange

In [3]:
class FeedForward(nn.Module):
  def __init__(self, dim, dim_mlp):
    super().__init__()
    # self.norm = nn.LayerNorm(dim)

    self.ffn = nn.Sequential(
        nn.LayerNorm(dim),
        nn.Linear(dim, dim_mlp),
        nn.GELU(),
        nn.Linear(dim_mlp, dim)
    )

  def forward(self, x):
    return self.ffn(x)

In [4]:
class SelfAttention(nn.Module):
  def __init__(self, dim, heads=8, dim_head=64):
    super().__init__()

    inner_feature = dim_head * heads
    self.heads = heads # FIX
    self.scale = dim_head ** -0.5
    self.attend = nn.Softmax(dim = -1)
    self.norm = nn.LayerNorm(dim)

    self.to_qkv = nn.Linear(dim, inner_feature * 3) # bias
    self.to_out = nn.Linear(inner_feature, dim)
  def forward(self, x, mask=None):
    x = self.norm(x)

    qkv = self.to_qkv(x)
    qkv = qkv.chunk(3, dim=-1)
    q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

    dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

    if mask is not None:
      dots = dots.masked_fill(mask, -1e9)

    dots = self.attend(dots)

    out = torch.matmul(dots, v)
    out = rearrange(out, 'b h n d -> b n (h d)')
    out = self.to_out(out)
    return out

In [5]:
class CasualMaskSelfAttention(nn.Module):
  def __init__(self, dim, heads=8, dim_head=64):
    super().__init__()

    inner_feature = dim_head * heads
    self.heads = heads
    self.scale = dim_head ** -0.5
    self.attend = nn.Softmax(dim = -1)
    self.norm = nn.LayerNorm(dim)

    self.to_qkv = nn.Linear(dim, inner_feature * 3) # bias
    self.to_out = nn.Linear(inner_feature, dim)
  def forward(self, x, pad_mask=None):
    b, n, _ = x.shape
    x = self.norm(x)

    qkv = self.to_qkv(x)
    qkv = qkv.chunk(3, dim=-1)
    q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), qkv)

    causal_mask = torch.triu(torch.ones((n, n), device=x.device), diagonal=1).bool()
    if pad_mask is not None:
        combined_mask = pad_mask | causal_mask
    else:
        combined_mask = causal_mask

    dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
    dots = dots.masked_fill(combined_mask, -1e9)
    dots = self.attend(dots)

    out = torch.matmul(dots, v)
    out = rearrange(out, 'b h n d -> b n (h d)')
    out = self.to_out(out)
    return out

In [6]:
class CrossAttention(nn.Module):
  def __init__(self, dim, heads=8, dim_head=64):
    super().__init__()

    inner_feature = dim_head * heads
    self.heads = heads
    self.scale = dim_head ** -0.5
    self.attend = nn.Softmax(dim = -1)
    self.norm = nn.LayerNorm(dim)

    self.to_q = nn.Linear(dim, inner_feature, bias=False)
    self.to_kv = nn.Linear(dim, inner_feature * 2, bias=False)

    self.to_out = nn.Linear(inner_feature, dim)
  def forward(self, x, enc, enc_mask=None):
    x = self.norm(x)
    enc = self.norm(enc)

    q = self.to_q(x)
    q = rearrange(q, 'b n (h d) -> b h n d', h = self.heads)

    kv = self.to_kv(enc)
    kv = kv.chunk(2, dim=-1)
    k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = self.heads), kv)

    dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
    if enc_mask is not None:
      dots = dots.masked_fill(enc_mask, -1e9)
    dots = self.attend(dots)

    out = torch.matmul(dots, v)
    out = rearrange(out, 'b h n d -> b n (h d)')
    out = self.to_out(out)
    return out

In [7]:
class Encoder(nn.Module):
  def __init__(self, dim, heads, dim_head, dim_mlp):
    super().__init__()
    self.attn = SelfAttention(dim, heads, dim_head)
    self.ffn = FeedForward(dim, dim_mlp)

  def forward(self, x, mask=None):
    x = self.attn(x, mask) + x
    x = self.ffn(x) + x
    return x

In [8]:
class Decoder(nn.Module):
  def __init__(self, dim, heads, dim_head, dim_mlp):
    super().__init__()

    self.mask_attn = CasualMaskSelfAttention(dim, heads, dim_head)
    self.cross_attn = CrossAttention(dim, heads, dim_head)
    self.ffn = FeedForward(dim, dim_mlp)

  def forward(self, x, enc, tgt_mask=None, enc_mask=None):
    x = self.mask_attn(x, pad_mask=tgt_mask) + x
    x = self.cross_attn(x, enc, enc_mask=enc_mask) + x
    x = self.ffn(x) + x
    return x

In [9]:
class Transformer(nn.Module):
  def __init__(self, dim, depth, heads, dim_head, dim_mlp, vocab_size, max_seq_len):
    super().__init__()

    self.position_encoding = nn.Parameter(torch.randn(1, max_seq_len, dim) * 0.02)
    self.layer = nn.ModuleList([])

    self.token_embedding_src = nn.Embedding(vocab_size, dim)
    self.token_embedding_tgt = nn.Embedding(vocab_size, dim)

    self.encoder_layers = nn.ModuleList([
      Encoder(dim, heads, dim_head, dim_mlp) for _ in range(depth)
    ])

    self.decoder_layers = nn.ModuleList([
      Decoder(dim, heads, dim_head, dim_mlp) for _ in range(depth)
    ])

    self.norm = nn.LayerNorm(dim)
    self.fc = nn.Linear(dim, vocab_size)
    # self.softmax = nn.Softmax(dim = -1)
  def forward(self, src, tgt):
    # shape: (batch, len, dim)
    src_len = src.size(1)
    tgt_len = tgt.size(1)

    src_mask = (src == 0).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, src_len)
    tgt_mask = (tgt == 0).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, tgt_len)

    src = self.token_embedding_src(src)
    tgt = self.token_embedding_tgt(tgt)

    src = src + self.position_encoding[:, :src_len, :]
    tgt = tgt + self.position_encoding[:, :tgt_len, :]

    for enc in self.encoder_layers:
      src = enc(src)

    for dec in self.decoder_layers:
      tgt = dec(tgt, src, tgt_mask=tgt_mask, enc_mask=src_mask)

    out = self.norm(tgt)
    return self.fc(out)

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-vi")

PAD_ID = tokenizer.pad_token_id
BOS_ID = tokenizer.eos_token_id
EOS_ID = tokenizer.eos_token_id

class TranslationDataset(Dataset):
    def __init__(self, split="train", max_seq_len=128):
        if split == "train":
          self.dataset = load_dataset("ura-hcmut/PhoMT", split="train[:500000]", trust_remote_code=True)
        if split == "validation":
          self.dataset = load_dataset("ura-hcmut/PhoMT", split="validation[:1000]", trust_remote_code=True)
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # Lấy cặp câu thô từ dataset
        item = self.dataset[idx]
        src_text = str(item["en"])
        tgt_text = str(item["vi"])

        # Tiền xử lý số hóa và đệm chuỗi (Padding/Truncation)
        src_inputs = tokenizer(src_text, padding='max_length', max_length=self.max_seq_len, truncation=True, return_tensors="pt")
        tgt_inputs = tokenizer(tgt_text, padding='max_length', max_length=self.max_seq_len, truncation=True, return_tensors="pt")

        # Squeeze để loại bỏ chiều batch dư thừa (1, seq_len) -> (seq_len)
        return {
            "src": src_inputs["input_ids"].squeeze(0),
            "tgt": tgt_inputs["input_ids"].squeeze(0)
        }

def preprocess_data(src_sentences, tgt_sentences, max_seq_len, device):
    """
    Biến đổi văn bản thô thành Tensor số nguyên có Padding hoàn chỉnh
    """
    src_inputs = tokenizer(src_sentences, padding='max_length', max_length=max_seq_len, truncation=True, return_tensors="pt")
    tgt_inputs = tokenizer(tgt_sentences, padding='max_length', max_length=max_seq_len, truncation=True, return_tensors="pt")

    src_tokens = src_inputs['input_ids'].to(device)
    tgt_tokens = tgt_inputs['input_ids'].to(device)

    return src_tokens, tgt_tokens

model = Transformer(dim=512, depth=6, heads=8, dim_head=64, dim_mlp=2048, vocab_size=tokenizer.vocab_size, max_seq_len=128).to(device)

def train_step(model, optimizer, criterion, src_sentences, tgt_sentences, max_seq_len, device):
    model.train()
    optimizer.zero_grad()

    # Tiền xử lý dữ liệu đầu vào
    src, tgt = preprocess_data(src_sentences, tgt_sentences, max_seq_len, device)

    # Teacher Forcing: Đầu vào Decoder dịch chuyển đi 1 token (bỏ token cuối)
    tgt_input = tgt[:, :-1]
    # Nhãn mục tiêu để tính Loss: dịch chuyển đi 1 token (bỏ token đầu [BOS])
    tgt_label = tgt[:, 1:]

    # Forward pass
    output = model(src, tgt_input) # output shape: (batch_size, tgt_len-1, vocab_size)

    # Flatten tensor để tính CrossEntropy Loss
    loss = criterion(output.reshape(-1, output.size(-1)), tgt_label.reshape(-1))

    loss.backward()
    optimizer.step()
    return loss.item()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [11]:
def translate_sentence(model, sentence, max_seq_len, device):
    """
    Hàm dịch tự hồi quy sử dụng chiến lược Greedy Search
    """
    model.eval()
    with torch.no_grad():
        # 1. Tiền xử lý câu nguồn (Thêm Pad, chuyển thành ID)
        src_inputs = tokenizer([sentence], padding='max_length', max_length=max_seq_len, truncation=True, return_tensors="pt")
        src = src_inputs['input_ids'].to(device)

        # 2. Khởi tạo câu đích ban đầu chỉ chứa duy nhất token bắt đầu [BOS]
        tgt_indices = [BOS_ID]
        tgt = torch.tensor([tgt_indices], dtype=torch.long, device=device) # Shape: (1, 1)

        # 3. Vòng lặp sinh tự hồi quy từng token một
        for _ in range(max_seq_len):
            # Chạy forward qua mạng Transformer
            output = model(src, tgt) # Shape: (1, current_tgt_len, vocab_size)

            # Lấy phân phối xác suất của token KẾ TIẾP (vị trí cuối cùng của chuỗi hiện tại)
            next_token_logits = output[:, -1, :]

            # Chọn từ có xác suất cao nhất (Greedy Decoding)
            next_token = next_token_logits.argmax(dim=-1).item()

            # Bổ sung từ vừa dự đoán vào chuỗi kết quả
            tgt_indices.append(next_token)
            tgt = torch.tensor([tgt_indices], dtype=torch.long, device=device)

            # Nếu gặp token kết thúc [EOS], dừng vòng lặp dịch ngay lập tức
            if next_token == EOS_ID:
                break

        # 4. Giải mã chuỗi ID số nguyên ngược lại thành văn bản chữ viết hoàn chỉnh
        translated_text = tokenizer.decode(tgt_indices, skip_special_tokens=True)
        return translated_text

In [18]:
from torchmetrics.text import BLEUScore

def evaluate_model(model, val_loader, tokenizer, device, max_gen_len=128):
    """
    Hàm quét qua tập dữ liệu kiểm thử, sinh câu dịch tự động và đánh giá điểm BLEU
    """
    model.eval()
    bleu = BLEUScore()

    references = []  # Chứa các câu dịch chuẩn (Ground Truth)
    hypotheses = []  # Chứa các câu do mô hình tự dịch

    # Định nghĩa ID token đặc biệt
    BOS_ID = tokenizer.eos_token_id
    EOS_ID = tokenizer.eos_token_id

    print("Đang tiến hành dịch và đánh giá trên tập Validation...")

    with torch.no_grad():
        for batch in val_loader:
            src = batch["src"].to(device)
            tgt = batch["tgt"].to(device)

            # Duyệt qua từng câu đơn lẻ trong batch để dịch tự hồi quy
            for i in range(src.size(0)):
                single_src = src[i:i+1] # Giữ nguyên shape (1, seq_len)

                # Giải mã câu đích chuẩn (Reference text) để đối chiếu
                ref_text = tokenizer.decode(tgt[i], skip_special_tokens=True)

                # Bắt đầu vòng lặp sinh từ tự hồi quy (Greedy Decode) cho câu hiện tại
                tgt_indices = [BOS_ID]
                tgt_tensor = torch.tensor([tgt_indices], dtype=torch.long, device=device)

                for _ in range(max_gen_len):
                    output = model(single_src, tgt_tensor)
                    next_token = output[:, -1, :].argmax(dim=-1).item()

                    tgt_indices.append(next_token)
                    tgt_tensor = torch.tensor([tgt_indices], dtype=torch.long, device=device)

                    if next_token == EOS_ID:
                        break

                # Giải mã câu hệ thống dịch (Hypothesis text)
                pred_text = tokenizer.decode(tgt_indices, skip_special_tokens=True)

                # Lưu kết quả (BLEU score nhận cấu trúc list của các token chuỗi văn bản)
                hypotheses.append(pred_text)
                references.append([ref_text]) # Mỗi hypothesis tương ứng với một list danh sách câu gốc
            break
    # Tính toán điểm số BLEU tổng quan
    bleu_score = bleu(hypotheses, references)
    print(f"\n--- Kết quả Đánh giá ---")
    print(f"Điểm BLEU trên tập Validation: {bleu_score.item() * 100:.2f}")

    # In thử nghiệm 3 cặp câu đầu tiên để kiểm tra trực quan
    for idx in range(min(3, len(hypotheses))):
        print(f"\n[Cặp câu {idx+1}]")
        print(f" - Chuẩn: {references[idx][0]}")
        print(f" - Model: {hypotheses[idx]}")

    return bleu_score.item()

In [13]:
def calculate_val_loss(model, val_loader, criterion, device):
    """
    Hàm tính toán giá trị Loss trung bình trên tập Validation
    """
    model.eval() # Chuyển mô hình sang chế độ đánh giá (tắt Dropout, Batch Normalization)
    total_val_loss = 0.0

    with torch.no_grad(): # Tắt tính toán gradient để tiết kiệm bộ nhớ và tăng tốc
        for batch in val_loader:
            src = batch["src"].to(device) # (batch_size, src_len)
            tgt = batch["tgt"].to(device) # (batch_size, tgt_len)

            # Mô hình dịch máy học dịch bằng cách dự đoán từ tiếp theo (Teacher Forcing)
            # Đầu vào của Decoder bỏ từ cuối cùng
            decoder_input = tgt[:, :-1]
            # Đầu ra mục tiêu (Ground Truth) bỏ từ đầu tiên (BOS token)
            targets = tgt[:, 1:]

            # Chạy forward pass
            outputs = model(src, decoder_input) # (batch_size, tgt_len-1, vocab_size)

            # Flatten tensor để tính Cross Entropy Loss
            outputs = outputs.reshape(-1, outputs.size(-1))
            targets = targets.reshape(-1)

            # Tính loss của batch hiện tại
            loss = criterion(outputs, targets)
            total_val_loss += loss.item()

    # Tính Loss trung bình trên toàn bộ tập Validation
    avg_val_loss = total_val_loss / len(val_loader)
    return avg_val_loss

In [14]:
BATCH_SIZE = 64
max_seq_len = 128
print(tokenizer.vocab_size)

train_dataset = TranslationDataset(split="train", max_seq_len=max_seq_len)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TranslationDataset(split="validation", max_seq_len=max_seq_len)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = Transformer(dim=512, depth=6, heads=8, dim_head=64, dim_mlp=1024, vocab_size=tokenizer.vocab_size, max_seq_len=max_seq_len)
model = model.to(device)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ura-hcmut/PhoMT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ura-hcmut/PhoMT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


53685


README.md:   0%|          | 0.00/653 [00:00<?, ?B/s]

PhoMT_training.csv:   0%|          | 0.00/525M [00:00<?, ?B/s]

PhoMT_validation.csv:   0%|          | 0.00/3.27M [00:00<?, ?B/s]

PhoMT_test.csv:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2977999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/18720 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19151 [00:00<?, ? examples/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ura-hcmut/PhoMT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ura-hcmut/PhoMT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


In [15]:
from tqdm import tqdm
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
# ignore_index giúp hàm Loss bỏ qua, không tính toán sai lệch trên các token đệm [PAD]

epochs = 5
for epoch in range(epochs):
    total_loss = 0
    for batch_idx, batch in tqdm(enumerate(train_loader)):
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        # Tách cấu trúc Teacher Forcing trực tiếp trên Tensor
        tgt_input = tgt[:, :-1]
        tgt_label = tgt[:, 1:]

        model.train()
        optimizer.zero_grad()

        output = model(src, tgt_input)
        loss = criterion(output.reshape(-1, output.size(-1)), tgt_label.reshape(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(train_loader)} | Train Loss: {loss.item():.4f} | Val Loss: {calculate_val_loss(model, val_loader, criterion, device)}")

    print(f"--- Kết thúc Epoch {epoch+1} | Loss train: {total_loss / len(train_loader):.4f} ---")
    score = evaluate_model(model, val_loader, tokenizer, device, max_gen_len=max_seq_len)


1it [00:07,  7.42s/it]

Epoch 1/5 | Batch 0/7813 | Train Loss: 11.1238 | Val Loss: 10.41489964723587


101it [02:00,  2.93s/it]

Epoch 1/5 | Batch 100/7813 | Train Loss: 3.4215 | Val Loss: 3.6302457749843597


201it [03:52,  2.95s/it]

...

Epoch 1/5 | Batch 3100/7813 | Train Loss: 1.6481 | Val Loss: 1.9614716321229935


3201it [1:00:01,  2.92s/it]

Epoch 1/5 | Batch 3200/7813 | Train Loss: 1.6563 | Val Loss: 1.9451859444379807


3276it [1:01:21,  1.12s/it]


KeyboardInterrupt: 

In [19]:
score = evaluate_model(model, val_loader, tokenizer, device, max_gen_len=max_seq_len)

Đang tiến hành dịch và đánh giá trên tập Validation...

--- Kết quả Đánh giá ---
Điểm BLEU trên tập Validation: 0.92

[Cặp câu 1]
 - Chuẩn: Vào chủ nhật ngày 1-9-2019, cơn bão Dorian, một trong những cơn bão mạnh nhất được ghi nhận ở Đại Tây Dương, với sức gió 362 km/h đổ bộ vào đảo Great Abaco, miền bắc Bahamas.
 - Model: , một trong những những những những những căn bản đất điểm được đường đầy điểm, đất làm từ đất, đã đưa ra 1 vào đất điểm được điểm được điểm được điểm được điều đó.

[Cặp câu 2]
 - Chuẩn: Bão Dorian đặc biệt nguy hiểm vì nó di chuyển chậm, có tốc độ gió cao và gây mưa lớn.
 - Model: làm tăng của nguyên nhân tốc động, đặc biệt làm tăng lên và động lớn.

[Cặp câu 3]
 - Chuẩn: Khi đi qua quần đảo Leeward, Puerto Rico và quần đảo Virgin, cơn bão này suy yếu thành áp thấp nhiệt đới nên gần như không gây thiệt hại gì.
 - Model: , đưa ra lên lá cảnh và điện thoại của bằng cách điện thoại, và không có một cách tác dụng để tăng tính toán học.
